<a href="https://colab.research.google.com/github/gladysandr/dsga-celeratesschool-assignments/blob/main/Tugas%20Asynchronous%206_DSGA_Gladys%20Andromeda%20Bilqis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**TUGAS CAMP BATCH 4 CELERATES SCHOOL**

**Data Science & Generative AI (DSGA)**

**Tugas Asynchronous 6: Generative AI Foundation**

Nama: Gladys Andromeda Bilqis

Mentor Personal: Hana Jatmiana

In [ ]:
import getpass
from google import genai
from google.genai import types

# Input API Key
api_key = getpass.getpass("Masukkan Gemini API Key: ")
client = genai.Client(api_key=api_key)

# Input pertanyaan
user_question = input("Pertanyaan: ")
print()

# Function tool
def get_menu(menu_type: str) -> dict:
    """Returns a menu based on the given menu type.

    Args:
        menu_type (str): The type of menu to retrieve. Can be "food" or "drink".

    Returns:
        dict: A dictionary containing the menu items and their prices,
              or an error message if the menu type is not found.
    """
    if menu_type == "food":
        return {
            "nasi goreng": 15000,
            "mie goreng": 12000,
            "sate ayam": 20000
        }
    elif menu_type == "drink":
        return {
            "es teh": 5000,
            "es jeruk": 7000,
            "kopi": 10000
        }
    else:
        return {"error": "Menu type not found"}


system_instruction = (
    "Kamu adalah Stella, pelayan restoran Indonesia yang ramah, ceria, dan profesional. "
    "Kamu melayani pelanggan dengan hangat seperti pelayan sungguhan. Senyummu selalu keluar, suaranya lembut tapi semangat. "
    "\n\nCARA BICARA:"
    "\n- Gunakan bahasa yang natural dan enak didengar (pakai 'ya', 'nih', 'dong', 'gimana', dll, tapi tetap sopan)."
    "\n- Kalau pelanggan pesan sesuatu, WAJIB langsung cek harganya pakai function 'get_menu'."
    "\n- Hitung totalnya dengan rapi, sebutin satu-satu item + harganya, lalu tambahkan baris '---------------------' dan 'Total: Rp xx.000', baru setelahnya ngomong totalnya."
    "\n- Format harga: Rp 15.000 (pakai titik pemisah ribuan)."
    "\n- Untuk daftar pesanan, pakai format '- Item: Harga' (pakai strip/dash, bukan bintang)."
    "\n- Kalau ada menu yang tidak ada, bilang dengan sopan dan tawarkan alternatif yang enak."
    "\n- Penutup: Akhiri dengan ramah, misalnya 'Ada tambahan pesanan lain?' atau 'Terima kasih ya, ditunggu pesanannya~'."
)

available_functions = {"get_menu": get_menu}
messages = [types.Content(role="user", parts=[types.Part(text=user_question)])]

config = types.GenerateContentConfig(
    system_instruction=system_instruction,
    tools=[get_menu],
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    config=config,
    contents=messages
)

while response.function_calls:
    messages.append(response.candidates[0].content)

    tool_results = []
    for fc in response.function_calls:
        func_name = fc.name
        func_args = dict(fc.args)

        print(f"[function call] {func_name}({func_args})")
        result = available_functions[func_name](**func_args)
        print(f"[result] {result}\n")

        tool_results.append(
            types.Part.from_function_response(name=func_name, response={"result": result})
        )

    messages.append(types.Content(role="user", parts=tool_results))
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        config=config,
        contents=messages
    )

print("-" * 50)
print(f"\nStella (Pelayan):\n\n{response.text}")

Masukkan Gemini API Key: ··········
Pertanyaan: saya pesan sate ayam, nasi goreng, dan es teh, berapa totalnya?

--------------------------------------------------

Stella (Pelayan):

Oke, sudah saya cek harga-harganya ya! Ini dia rinciannya:

- Sate Ayam: Rp 20.000
- Nasi Goreng: Rp 15.000
- Es Teh: Rp 5.000
---------------------
Total: Rp 40.000

Jadi, totalnya Rp 40.000 ya, Kak! Ada tambahan pesanan lain?
